# Effect of Label Noise on Model Accuracy and Mitigation Strategies
### Comparative Study: Logistic Regression, Random Forest, and XGBoost

This notebook investigates the degradation dynamics of supervised machine learning models when exposed to increasing symmetric label noise, and evaluates recovery mitigations via **Data Cleaning** (Out-of-Fold Cross-Validation Filtering) and **Noise-Aware Training** (Sample Reweighting & Regularization).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.experiment import load_dataset, run_noise_degradation_study, run_mitigation_benchmark
from src.noise_injector import LabelNoiseInjector
from src.models import get_model
from src.mitigation import OutOfFoldCleaner, compute_soft_sample_weights
from src.visualization import plot_accuracy_degradation, plot_mitigation_comparison, plot_decision_boundaries_2d

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Theoretical Hypotheses

- **Hypothesis 1 (Logistic Regression)**: As a generalized linear model with low capacity (constrained VC-dimension), Logistic Regression cannot fit local noise anomalies. However, as noise increases symmetrically, the true gradient expectation $\mathbb{E}[\nabla_w \mathcal{L}]$ shrinks toward zero, flattening the sigmoid and increasing classification boundary variance.
- **Hypothesis 2 (Random Forest)**: Bagging buffers label noise at low rates ($\le 20\%$) because mislabeled samples only appear in $\approx 63.2\%$ of bootstrap bags, and deep tree splits on noise are outvoted by clean tree consensus. Above $30\%$, majority voting breaks down.
- **Hypothesis 3 (XGBoost)**: Gradient boosting constructs additive trees sequentially by fitting to the negative gradients (pseudo-residuals). Because mislabeled instances produce extreme residual errors ($|y_i - p_i| \approx 1$), boosting aggressively concentrates tree splits on noisy outliers, suffering the sharpest degradation unless explicitly constrained by subsampling and shrinkage.

In [ ]:
# Load Wisconsin Breast Cancer Dataset
X, y = load_dataset('breast_cancer')
print(f"Dataset Shape: {X.shape}, Class Distribution: {np.bincount(y)}")

## 2. Load Precomputed Degradation Results
We analyze the empirical degradation across noise rates $[0\%, 5\%, 10\%, 20\%, 30\%, 40\%, 50\%]$.

In [ ]:
df_degradation = pd.read_csv('../results/noise_degradation_metrics.csv')
summary = df_degradation.groupby(['model', 'noise_rate'])['accuracy'].agg(['mean', 'std']).reset_index()
pivoted = summary.pivot(index='noise_rate', columns='model', values='mean')
print("Mean Test Accuracy vs. Noise Level:")
display(pivoted.map(lambda v: f"{v*100:.1f}%"))

In [ ]:
# Display Accuracy vs Noise Curve
from IPython.display import Image
Image(filename='../results/figures/accuracy_vs_noise.png')

## 3. Mitigation Strategies at 30% Noise
Evaluating:
1. **Clean Baseline**
2. **Unmitigated Noisy**
3. **Robust Regularization**
4. **Data Cleaning (OOF Filter)**
5. **Soft Noise-Aware Sample Reweighting**

In [ ]:
df_mitigation = pd.read_csv('../results/mitigation_results.csv')
piv_mit = df_mitigation.pivot(index='model', columns='strategy', values='accuracy')
display((piv_mit * 100).round(2).astype(str) + '%')

In [ ]:
Image(filename='../results/figures/mitigation_comparison.png')

## 4. 2D Decision Boundary Deformations
Visualizing how label noise distorts non-linear decision boundaries on the Moons dataset, and how mitigations recover smooth decision frontiers.

In [ ]:
Image(filename='../results/figures/decision_boundaries.png')